<a href="https://colab.research.google.com/github/aishwaryasuriyakumar/aishwaryasrcs09-codeboosters-internship-2026/blob/main/Day_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tool Calling:
get_schema()
generate_sql(question)
execute_sql(query)

Multi-Step Reasoning:The ReAct Pattern
1.Think
2.Plan
3.Act
4.Respond

The Complete Text-to-SQL Pipeline
Load CSV into SQLite ->

In [ ]:
!pip install groq -q

print("installed Successfully!")

installed Successfully!


In [ ]:
import sqlite3

import pandas as pd

import os

from groq import Groq

import re

print("All libraries are imported successfully ")

All libraries are imported successfully 


In [ ]:
import os
os.environ["Groq_API_KEY"] = "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"

client = Groq(api_key=os.environ["Groq_API_KEY"])
MODEL = "llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using model : {MODEL}")

Groq client initialized successfully
Using model : llama-3.1-8b-instant


In [ ]:
import os
import pandas as pd
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
df=pd.read_csv(io.StringIO(csv_data))
print(f"Dataset loaded:{len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows:")
df.head()

Dataset loaded:30 rows, 8 columns

First 5 rows:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
import sqlite3
conn= sqlite3.connect("college.db")
df.to_sql("students", conn, if_exists="replace",index=False)
print("Database created: college.db")
print("Table 'students' created with 30 student records")
test_df=pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"\nVerification:{test_df['total_rows'][0]} rows in database")

Database created: college.db
Table 'students' created with 30 student records

Verification:30 rows in database


In [ ]:
#function to get the database schema
def get_schema(conn,table_name="students"):
  """
  This function reads the structure of a database table.
  It returns information about each column:name and data type.
  Parameters:
  conn: The SQLite connection object(our link to the database)
  table_name: The name of the table to inspect(default:'students')
  Returns:
      A formatted string describing the table's structure
  """
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"Table:{table_name}"]
  schema_lines.append("Columns:")
  for col in columns:
    schema_lines.append(f" - {col[1]} ({col[2]})")
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()
  schema_lines.append("Sample Rows:")
  for row in sample_rows:
    schema_lines.append(f" - {row}")
  return "\n".join(schema_lines)

schema = get_schema(conn)
print(schema)

Table:students
Columns:
 - student_id (INTEGER)
 - name (TEXT)
 - age (INTEGER)
 - gender (TEXT)
 - subject (TEXT)
 - marks (INTEGER)
 - attendance (INTEGER)
 - grade (TEXT)
Sample Rows:
 - (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 - (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 - (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
from groq import Groq
import os

def generate_sql(user_question,schema_text,client,model):
  """
  Generates an SQL query based on a user question and database schema.
  """
  system_message = f"""You are an expert SQL assistant, you are connected to a sqlite database with the following structure:
  {schema_text}
  rules you must follow:
  1. Generate ONLY a valid sqlite query.
  2. Don't include any explanation or text, only the SQL query.
  3. Don't use markdown code blocks. Return the raw SQL query.
  4. The table name is: students.
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses (e.g.: WHERE name='john').
  7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """
  try:
    response = client.chat.completions.create(
      model=model,
      messages=[
        {"role":"system","content":system_message},
        {"role":"user","content":user_question},
      ],
      temperature =0.0
    )
    sql_query= response.choices[0].message.content.strip()
  except Exception as e:
    print(f"Error generating SQL query: {e}")
    print("Please check your Groq API key and ensure it is valid and has no restrictions.")
    sql_query = "" # Assign an empty string to prevent NameError in subsequent cells
  return sql_query

question ="show me all the female students"
print(f"Question: {question}")
sql_query = generate_sql(question,schema,client,MODEL)
print(f"\nSQL Query:\n{sql_query}")

Question: show me all the female students

SQL Query:
SELECT * FROM students WHERE gender='Female'


In [ ]:
import re
import pandas as pd # Ensuring pandas is imported for read_sql_query

def execute_sql(sql_query,conn):
  """
  Executes an SQL query against the database and returns a pandas DataFrame.
  """
  clean_sql = sql_query.strip()
  clean_sql = re.sub(r'```sql\s*','',clean_sql)
  try:
    result_df = pd.read_sql_query(clean_sql,conn)
    return result_df,None
  except Exception as e:
    return None,str(e)

# Assuming 'sql_query' is available from the execution of the previous cell (generate_sql)
# and 'conn' is available from earlier database setup.
# Note: The 'generate_sql' function in cell pQGGmCwMDljo should use 'prompt' instead of 'system_prompt' for correct functioning.

print(f"Executing SQL:\n{sql_query}")
result,error = execute_sql(sql_query,conn)
if error:
  print(f"Error: {error}")
else:
  print(f"\nQuery returned {len(result)} rows.")
  print(result.head())

Executing SQL:
SELECT * FROM students WHERE gender='Female'

Query returned 15 rows.
   student_id            name  age  gender      subject  marks  attendance  \
0           2     Priya Patel   21  Female      Science     76          85   
1           4      Sneha Iyer   22  Female  Mathematics     62          78   
2           6  Divya Krishnan   20  Female      Science     83          88   
3           8    Ananya Gupta   21  Female  Programming     89          96   
4          10    Pooja Sharma   22  Female  Mathematics     55          72   

  grade  
0     B  
1     C  
2     A  
3     A  
4     D  


In [ ]:
def text_to_sql_agent(user_question, conn, client , model , verbose=True):
  print("=" * 60)
  print(f"User Question: {user_question}")
  print("=" * 60)

  if verbose:
    print("\n[STEP 1] Reading database schema...")

  schema_text = get_schema(conn)

  if verbose:
    print("Schema loaded successfully")

  if verbose:
    print("\n[STEP 2] Generating SQL query with Groq LLM...")

  generated_sql = generate_sql(user_question, schema_text, client, model)

  if verbose:
    print(f"Generated sql:\n {generated_sql}")

  if verbose:
    print("\n[STEP3] Executing SQL on the database...")

  result_df, error=execute_sql(generated_sql,conn)

  if error:
    print(f"SQL Execution error : {error}")
    return None, generated_sql

  if verbose:
    print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print("-" *40)
    print(result_df.to_string(index=False))
    print("=" * 60)

    return result_df , generated_sql

result, sql_used = text_to_sql_agent(
     "Show top 5 students  in Programming",
     conn,client,MODEL
 )

User Question: Show top 5 students  in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT name, marks, attendance, grade FROM students WHERE subject='Programming' ORDER BY marks DESC LIMIT 5

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
        name  marks  attendance grade
Aditya Kumar     97          99    A+
 Rohan Mehta     95          98    A+
Anjali Singh     94          97    A+
 Nandita Rao     92          96    A+
  Arjun Nair     91          94    A+


In [ ]:
result1,_=text_to_sql_agent(
"show me all students who study Mathematics",
conn,client,MODEL
)

User Question: show me all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT * FROM students WHERE subject='Mathematics'

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Fema

In [ ]:
result2,_=text_to_sql_agent(
    "What is the average marks for each subject?",
    conn, client, MODEL
)

User Question: What is the average marks for each subject?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT subject, AVG(marks) FROM students GROUP BY subject

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 3 row(s)

RESULTS:
----------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4


In [ ]:
result3,_=text_to_sql_agent(
    "Show students who scored more than 90 mark",
    conn,client,MODEL
)

User Question: Show students who scored more than 90 mark

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT * FROM students WHERE marks > 90

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 6 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+


In [ ]:
result4,_=text_to_sql_agent(
    "How many male and female students are there?",
    conn,client,MODEL

)

User Question: How many male and female students are there?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 male_count  female_count
         15            15


In [ ]:
from operator import ge
import pandas as pd # Added import for DataFrame operations
from IPython.display import display # Added import for displaying DataFrames

def generate_answer(user_question, query_results, client, model):
  """
  Generates a natural language answer based on user question and query results.
  """
  # query_results is expected to be a DataFrame here
  if query_results is None or len(query_results) == 0:
    return "NO results were found for your query."
  results_text = query_results.to_string(index=False)
  prompt = f"""The user asked:'{user_question}'
The database return these results:
{results_text}
Please write a clear, friendly, 2-3 sentence answer to the question based on these results.
Be specific. Mention actual names and numbers form the data. Do not add information not present in the results."""
  try:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()
  except Exception as e:
    return f"Error generating natural language answer: {e}"

def smart_text_to_sql_agent(user_question, conn, client, model):
  print("=" * 60)
  print(f"User Question: {user_question}")
  print("=" * 60)

  schema_text = get_schema(conn)
  print("Generating Sql..")

  # This calls the 'generate_sql' from cell YJj8rDVzdYML to get the SQL query
  generated_sql = generate_sql(user_question, schema_text, client, model)
  print(f"Sql:{generated_sql}")

  if not generated_sql:
    print("Error: SQL query generation failed or returned an empty query.")
    return None, ""

  print("Executing SQL on the database..")
  result_df, error = execute_sql(generated_sql, conn)
  if error:
    print(f"Error executing SQL: {error}")
    return None, generated_sql

  print(f"\nData({len(result_df)} rows returned):")
  display(result_df)

  print("\ngenerating natural language answer..")
  # This calls the local 'generate_answer' function defined above
  answer = generate_answer(user_question, result_df, client, model)
  print(f"\nAnswer:{answer}")
  print("="*60)

  return result_df, generated_sql

# Corrected function call with all required arguments
smart_text_to_sql_agent("who are the top 5 students", conn, client, MODEL)

User Question: who are the top 5 students
Generating Sql..
Sql:SELECT name FROM students ORDER BY marks DESC LIMIT 5
Executing SQL on the database..

Data(5 rows returned):


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Bhavna Mehta
4,Nandita Rao



generating natural language answer..

Answer:Based on the results, the top 5 students are: 

1. Aditya Kumar
2. Rohan Mehta
3. Anjali Singh
4. Bhavna Mehta
5. Nandita Rao


(           name
 0  Aditya Kumar
 1   Rohan Mehta
 2  Anjali Singh
 3  Bhavna Mehta
 4   Nandita Rao,
 'SELECT name FROM students ORDER BY marks DESC LIMIT 5')